# Evolve your harness

One full pass of the harness evolution loop, cell by cell: serve Reef,
send three coding tasks through it, report each score, and watch one
gated evolve step improve a skill and publish a new harness version.

On Colab, pick a GPU runtime (T4 is enough) and run the setup cell below;
it installs everything and serves the model. A local run instead needs
Reef installed in the kernel's environment (`pip install -e .` from a
checkout, which brings `reef-client`), the `git-lfs` system package,
the `pi` binary on PATH
(`npm i -g @earendil-works/pi-coding-agent@0.84.2`), and an
OpenAI-compatible endpoint serving a model. Set the three values in the
next cell. No GPU is needed by Reef itself.

[<img align="left" src="https://colab.research.google.com/assets/colab-badge.svg">](https://colab.research.google.com/github/Human-Agent-Society/reef/blob/main/tutorials/evolve-your-harness/evolve-your-harness.ipynb)

In [1]:
# Colab setup: clone the repo, install Reef and the pi agent, and serve a
# model with ollama (pick a GPU runtime; T4 is enough). A run from a local
# checkout skips this cell.
import importlib.util
import os
import pathlib
import subprocess
import sys

# find_spec("google.colab") raises outside Colab when no google namespace package is installed.
on_colab = importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None
if on_colab:
    root = pathlib.Path("/content/reef")
    if not root.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/Human-Agent-Society/reef", str(root)], check=True)
    os.chdir(root / "tutorials" / "evolve-your-harness")

    missing_git_lfs = subprocess.run(
        ["git", "lfs", "version"], capture_output=True
    ).returncode != 0
    missing_ollama = subprocess.run(
        ["which", "ollama"], capture_output=True
    ).returncode != 0
    packages = []
    if missing_git_lfs:
        packages.append("git-lfs")
    if missing_ollama:
        packages.append("zstd")
    if packages:
        subprocess.run(["apt-get", "update", "-qy"], check=True)
        subprocess.run(["apt-get", "install", "-qy", *packages], check=True)
    if missing_git_lfs:
        subprocess.run(["git", "lfs", "install"], check=True)

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(root)], check=True)
    subprocess.run(["npm", "install", "-g", "--silent",
                    "@earendil-works/pi-coding-agent@0.84.2"], check=True)
    if missing_ollama:
        # A pinned release asset, extracted directly: install.sh fails in a
        # container (its service-manager setup has nothing to manage), and the
        # ollama.com tarball alias 404s upstream since the release assets
        # moved to zstd.
        subprocess.run("curl -fSL https://github.com/ollama/ollama/releases/download/"
                       "v0.33.2/ollama-linux-amd64.tar.zst"
                       " | tar --zstd -x -C /usr/local", shell=True, check=True)
    subprocess.run("nohup ollama serve >/tmp/ollama.log 2>&1 &", shell=True, check=True)
    import time
    import urllib.request
    for _ in range(30):
        try:
            urllib.request.urlopen("http://127.0.0.1:11434", timeout=1).close()
            break
        except OSError:
            time.sleep(1)
    else:
        raise RuntimeError("ollama did not start within 30s; see /tmp/ollama.log")
    subprocess.run(["ollama", "pull", "qwen2.5:7b"], check=True)
    print("colab setup complete")


In [2]:
# The model under test: any OpenAI-compatible endpoint (no /v1 suffix).
UPSTREAM_URL = "http://127.0.0.1:11434"
UPSTREAM_MODEL = "qwen2.5:7b"
UPSTREAM_API_KEY = "sk-local"


## Start Reef

`serve.yaml` carries the whole stack: the service and the
`harness_evolve` recipe sections (seed skill composition, the three
evolve tasks, the gate). We parse its schema-v2 structure, patch the
inference provider and model binding to the values above, and use the same
recipe materializer as `run.sh` before starting the service as a subprocess.
Reef is a plain Python process; the model stays wherever it is served.

In [3]:
import os
import pathlib
import subprocess
import sys
import time
import urllib.request

import yaml

from harness.materialize_recipe import materialize

here = pathlib.Path.cwd()
work = here / "work"

cfg = yaml.safe_load((here / "configs" / "serve.yaml").read_text())
cfg["inference"]["upstream-url"] = UPSTREAM_URL
cfg["inference"]["upstream-model"] = UPSTREAM_MODEL
config_path = work / "serve-notebook.yaml"
config_path.parent.mkdir(parents=True, exist_ok=True)
config_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
materialize(config_path, work)

env = dict(os.environ)
env["REEF_RECIPE_CONFIG_DIR"] = str(work / "recipes")
env["REEF_UPSTREAM_API_KEY"] = UPSTREAM_API_KEY
env["PYTHONPATH"] = str(here.parent.parent)

serve = subprocess.Popen(
    [sys.executable, "-m", "reef", "serve", "-c", str(config_path)],
    env=env, stdout=(work / "reef.log").open("w"), stderr=subprocess.STDOUT,
)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:8900/healthz", timeout=1).close()
        break
    except OSError as exc:
        if serve.poll() is not None:
            raise RuntimeError((work / "reef.log").read_text()[-2000:]) from exc
        time.sleep(1)
else:
    serve.terminate()
    serve.wait(timeout=60)  # let the stack go before a rerun binds the port again
    raise RuntimeError("Reef did not start within 60s; see work/reef.log")
print("reef is up")


reef is up


## Send the tasks and report the scores

Each task goes through Reef's inference proxy, which injects the served
skill catalog and records the exchange. The report ties the score to
that recorded traffic. Only failing reports batch; the first failure
schedules one evolve step.

In [4]:
import json
from reef_client import ReefClient, ReefClientError
from harness import evolution

SCENARIO = "harness-evolve-demo"
client = ReefClient("http://127.0.0.1:8900", token="reef-local", timeout_s=300.0)
tasks = json.loads((work / "tasks.json").read_text())


def training_steps():
    """The catalog's training rows, oldest first; each carries one evolve step's verdict and metrics."""
    try:
        rows = client.get("/reef/harness/releases", extra_headers={"x-reef-scenario": SCENARIO})["releases"]
    except ReefClientError as exc:
        if exc.status != 404:  # 404: nothing has named the scenario yet
            raise
        return []
    return [row for row in rows if row.get("operation") == "training"]


before = len(training_steps())
failures = 0
for index, task in enumerate(tasks, start=1):
    body, receipt = client.inference_with_record(
        SCENARIO, "/v1/chat/completions",
        {"model": UPSTREAM_MODEL, "messages": [{"role": "user", "content": task}]},
    )
    prefix = task.split(maxsplit=1)[0]
    score = evolution.grade_text(task, body["choices"][0]["message"]["content"])
    client.report(
        SCENARIO,
        {"agent_record_id": f"harness-evolve-{index}", "score": score, "feedback": prefix},
        references=[receipt],
    )
    failures += score == 0.0
    print(f"task {index} {prefix}: score {score}")
print(f"{failures} failing report(s); each schedules one gated evolve step"
      if failures else "every task passed: nothing batches, no evolve step runs")


task 1 [sieve]: score 1.0


task 2 [fib]: score 0.0


task 3 [csv]: score 1.0
1 failing report(s); each schedules one gated evolve step


## Watch the evolve step publish

The step proposes one skill mutation (the served model is its own
proposer), renders candidate and current compositions, runs the three
tasks twice each as headless `pi` episodes, and publishes only if the
candidate wins. The seed tree is served from the start: the step's verdict
lands as a training row in `GET /reef/harness/releases`, and a win moves
the head that `GET /reef/harness` serves.


In [5]:
# The manifest alone never says whether a step published, because the seed tree is served
# from the start: wait for the step's catalog row, then read the head a win left behind.
steps = training_steps()
deadline = time.monotonic() + 900
while len(steps) <= before and time.monotonic() < deadline and failures:
    if error := client.get("/reef/status").get("error"):
        raise SystemExit(f"evolve step failed: {error}")
    time.sleep(2)
    try:
        steps = training_steps()
    except (TimeoutError, OSError) as exc:
        # A step in flight holds the catalog until it ends; a service that is gone does not answer /healthz.
        try:
            client.get("/healthz")
        except (ReefClientError, TimeoutError, OSError):
            raise SystemExit("the service stopped answering; see work/reef.log") from exc
metrics = (steps[-1].get("metrics") or {}) if len(steps) > before else {}
if metrics.get("published"):
    manifest = client.get("/reef/harness", extra_headers={"x-reef-scenario": SCENARIO})
    print(f"published: artifact {manifest['release_id']}"
          f" (parent {manifest['parent_release_id']})")
    print(json.dumps(manifest["evaluation"], indent=2, sort_keys=True))
    for path, text in sorted(manifest["files"].items()):
        if "/skills/" in path:
            print(f"--- {path} ---\n{text}")
elif failures:
    # The step's row says why: a "skipped" metric means the proposer returned no
    # usable mutation; any other row means the candidate ran its episodes and lost.
    why = metrics.get("skipped") or ("rejected by the gate" if metrics else "no verdict within 900s")
    print(f"no publish this time ({why}); rerun the task cell for another attempt")
    print("proposal:", {key: (metrics.get("mutation") or {}).get(key) for key in ("op", "id")})
    print("gate:", {key: metrics.get(key) for key in ("wins", "losses", "ties", "candidate_score", "current_score")})


no publish this time (rejected by the gate); rerun the task cell for another attempt
proposal: {'op': 'update', 'id': 'answer-style'}
gate: {'wins': 0, 'losses': 2, 'ties': 1, 'candidate_score': 0.0, 'current_score': 1.0}


## Stop the service

The published tree stays under `work/`; any client can pull and install
it later through the HTTP API.

In [6]:
serve.terminate()
serve.wait(timeout=60)  # the service drains its executors for up to 30 s before it exits
print("reef stopped")


reef stopped
